In [1]:
import boto3
import os
from flask import Flask, jsonify, request
from flask_cors import CORS
from datetime import datetime, timedelta
import time
from data_generation import generate_cell_tower_data, generate_predicted_outages
import uuid
from utils import n_towers_within_zone
from dotenv import load_dotenv
from decimal import Decimal


In [2]:
load_dotenv()

# --- AWS DynamoDB setup ---
dynamodb = boto3.resource(
    "dynamodb",
    region_name="us-east-1", # change region if needed
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY"), # load from env variables
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)

# Change this to your DynamoDB table name
cell_towers_table = dynamodb.Table("cell-towers")
outages_table = dynamodb.Table("outages")

bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

In [3]:
# cell_tower_data = generate_cell_tower_data()

# for idx, tower_id in enumerate(cell_tower_data['tower_id']):
#     item = {
#         'tower_id': tower_id,
#         'status': cell_tower_data['status'][idx],
#         'longitude': Decimal(str(cell_tower_data['longitude'][idx])),
#         'latitude': Decimal(str(cell_tower_data['latitude'][idx])),
#         'signal_strength': Decimal(str(cell_tower_data['signal_strength'][idx])),
#         'coverage_radius': Decimal(str(cell_tower_data['coverage_radius'][idx])),
#         'bandwidth': cell_tower_data['bandwidth'][idx],
#         'technology': cell_tower_data['technology'][idx],
#     }
    
    # response = cell_towers_table.put_item(Item=item)


In [4]:
cell_towers_table.scan()['Items'][:3]

[{'technology': '5G',
  'coverage_radius': Decimal('4.468582477125174'),
  'signal_strength': Decimal('-109.42251809867219'),
  'tower_id': 'FN-1183',
  'status': 'Active',
  'longitude': Decimal('-100.60736108549787'),
  'latitude': Decimal('28.65929051830793'),
  'bandwidth': Decimal('20')},
 {'technology': '5G',
  'coverage_radius': Decimal('4.097806725732029'),
  'signal_strength': Decimal('-75.02288286462795'),
  'tower_id': 'FN-1021',
  'status': 'Active',
  'longitude': Decimal('-101.66059954850988'),
  'latitude': Decimal('29.045259681298685'),
  'bandwidth': Decimal('20')},
 {'technology': '4G LTE',
  'coverage_radius': Decimal('2.0966086361191953'),
  'signal_strength': Decimal('-93.91441519145724'),
  'tower_id': 'FN-1466',
  'status': 'Active',
  'longitude': Decimal('-94.10832817623255'),
  'latitude': Decimal('31.992331552453088'),
  'bandwidth': Decimal('80')}]

In [ ]:
outage_data = generate_predicted_outages(20)
cell_tower_data = cell_towers_table.scan()['Items']

severity_to_affected_towers = {
    'Critical': 0.5,
    'High': 0.25,
    'Medium': 0.1,
    'Low': 0.05
}

for idx, outage_id in enumerate(outage_data['outage_id']):
    item = {
        'outage_id': outage_id,
        'event': outage_data['event'][idx],
        'severity': outage_data['severity'][idx],
        'center_longitude': Decimal(str(outage_data['center_longitude'][idx])),
        'center_latitude': Decimal(str(outage_data['center_latitude'][idx])),
        'radius': Decimal(str(outage_data['radius'][idx]))
    }
    
    # get the number of towers within the zone
    towers_in_zone = n_towers_within_zone(outage_data['center_longitude'][idx], outage_data['center_latitude'][idx], outage_data['radius'][idx], cell_tower_data)
    towers_affected = int(towers_in_zone * severity_to_affected_towers[outage_data['severity'][idx]])
    
    item['towers_affected'] = towers_affected
    item['towers_total'] = towers_in_zone
    
    response = outages_table.put_item(Item=item)


In [9]:
len(outages_table.scan()['Items'])

20